# 混合并行  

混合并行（Hybrid Parallelism，HP）是将多种并行策略如数据并行、流水线并行和张量并行等进行混合使用。通过结合不同的并行策略，混合并行可以充分发挥各种并行策略的优点，以最大程度地提高计算性能和效率。针对千亿规模的大语言模型，通常在每个服务器内部使用张量并行策略，由于该策略涉及的网络通信量较大，需要利用服务器内部的不同计算设备之间的高速通信带宽。通过流水线并行，将模型的不同层划分为多个阶段，每个阶段由不同的机器负责计算。这样可以充分利用多台机器的计算能力，并通过机器之间的高速通信来传递计算结果和中间数据，以提高整体的计算速度和效率。最后，在外层叠加数据并行策略，以增加并发数量，提升整体训练速度。通过数据并行，将训练数据分发到多组服务器上进行并行处理，每组服务器处理不同的数据批次。这样可以充分利用多台服务器的计算资源，并增加训练的并发度，从而加快整体训练速度。  

BLOOM 使用了Megatron-DeepSpeed[108] 框架进行训练，主要包含两个部分：Megatron-LM 提供张量并行能力和数据加载原语；DeepSpeed[138] 提供ZeRO 优化器、模型流水线以及常规的分布式训练组件。通过这种方式可以实现数据、张量和流水线三维并行，BLOOM 模型训练时采用的并行计算结构如图4.14所示。BLOOM 模型训练使用了由48 个NVIDIA DGX-A100 服务器组成的集群，每个DGX-A100 服务器包含8 张NVIDIA A100 80GB GPU，总计包含384 张。BLOOM 训练采用的策略是首先将集群分为48 个一组，进行数据并行。接下来，模型整体被分为12 个阶段，进行流水线并行。每个阶段的模型被划分到4 张GPU 中，进行张量并行。同时BLOOM 也使用了ZeRO（零冗余优化器）[139] 进一步降低了模型对显存的占用。用了通过上述四个步骤可以实现数百个GPU 的高效并行计算。 

![](images/ec04c1a43b0567cc94e8d4281d7d9e0549fccc672758c4645fdee2f4535d290f.jpg)  

In [ ]:
# 